# Structural Patterns

## What's covered

- The shared theme of the structural family — **composition over inheritance**
- **Adapter** — make a class with the wrong interface usable through the right one
- **Bridge** — split an abstraction from its implementation so both can vary independently
- **Composite** — treat individual objects and groups of objects uniformly
- **Decorator** — add behaviour to an object without changing its class
- **Facade** — a simplified entry point to a complex subsystem
- **Flyweight** — share fine-grained objects to save memory
- **Proxy** — a stand-in that controls access to another object


## The shared theme — composition over inheritance

The seven structural patterns are all answers to the same question: **how do you assemble small objects into a larger structure?** The naive answer — inheritance — runs out of headroom fast. A class hierarchy grows linearly in the number of variations: every new behaviour becomes a new subclass, every new dimension of variation multiplies the count. Composition scales better because you wire objects together at runtime rather than locking the structure into the class hierarchy at compile time.

Each of the seven patterns picks a different recurring composition shape and names it.

- **Adapter** and **Facade** are about *interface translation* — making one shape usable as another.
- **Bridge** and **Composite** are about *splitting structure* into two dimensions that vary independently.
- **Decorator** is about *layering behaviour* — wrapping an object to add a transparent capability.
- **Flyweight** and **Proxy** are about *controlling access* — sharing objects, or standing in for them.

A useful contrast with the creational family from notebook two: creational patterns answer "who decides which object to build?"; structural patterns answer "how do the built objects fit together?"


## Adapter

**Intent:** convert the interface of one class into another that the client expects.

**Where it shows up:** wrapping a third-party library so it conforms to your domain's interface, integrating legacy code without rewriting it, swapping an old API for a new one without changing every call site.

**The shape:** the adapter holds a reference to the *adaptee* (the class with the wrong interface) and implements the *target* interface (the one the client expects). Calls on the target are translated into calls on the adaptee.

**Python's take:** duck typing often makes adaptation transparent — if an object has the methods the caller invokes, it works, regardless of its declared type. An explicit adapter is still useful when method names or argument shapes differ.

**Kotlin's take:** extension functions can serve as one-way adapters — adding a method that calls into the existing API in the right shape. For more involved translation, a wrapper class.


### Python


In [ ]:
# Third-party payment API — we don't control this shape.
class LegacyStripeApi:
    def charge_card(self, amount_cents: int, card_token: str) -> dict:
        return {"status": "ok", "id": "ch_123", "amount": amount_cents}


# Our application speaks in transactions, not card charges.
from dataclasses import dataclass


@dataclass
class Transaction:
    amount: float          # in dollars
    currency: str
    payment_method: str    # opaque token


class PaymentGateway:
    # The interface our app expects every gateway to satisfy.
    def process(self, txn: Transaction) -> str: ...


# The adapter — translates our shape into the legacy library's shape.
class StripeAdapter(PaymentGateway):
    def __init__(self, api: LegacyStripeApi):
        self._api = api

    def process(self, txn: Transaction) -> str:
        if txn.currency != "USD":
            raise ValueError("legacy Stripe API only handles USD")
        cents = int(txn.amount * 100)
        result = self._api.charge_card(cents, txn.payment_method)
        return result["id"]


gateway: PaymentGateway = StripeAdapter(LegacyStripeApi())
print(gateway.process(Transaction(amount=42.50, currency="USD", payment_method="tok_abc")))
# ch_123


### Kotlin

```kotlin
class LegacyStripeApi {
    fun chargeCard(amountCents: Int, cardToken: String): Map<String, Any> =
        mapOf("status" to "ok", "id" to "ch_123", "amount" to amountCents)
}

data class Transaction(val amount: Double, val currency: String, val paymentMethod: String)

interface PaymentGateway {
    fun process(txn: Transaction): String
}

class StripeAdapter(private val api: LegacyStripeApi) : PaymentGateway {
    override fun process(txn: Transaction): String {
        require(txn.currency == "USD") { "legacy Stripe API only handles USD" }
        val cents = (txn.amount * 100).toInt()
        val result = api.chargeCard(cents, txn.paymentMethod)
        return result["id"] as String
    }
}

val gateway: PaymentGateway = StripeAdapter(LegacyStripeApi())
println(gateway.process(Transaction(42.50, "USD", "tok_abc")))
```


**When not to use Adapter.** If you control both sides, change one of them directly. The pattern exists because you do *not* control one side — usually the third-party library or the legacy API. If the interfaces are almost the same and duck typing covers the gap (Python especially), skip the explicit adapter.


## Bridge

**Intent:** decouple an abstraction from its implementation so that both can vary independently.

**Where it shows up:** anywhere you have *two orthogonal dimensions of variation* and inheritance would force you into a class explosion. Cross-platform rendering (shapes × renderers), database access (queries × backends), notifications (messages × delivery channels).

**The problem it solves:** suppose you have three shapes (Circle, Square, Triangle) and two renderers (Vector, Raster). With inheritance, you end up with six classes — `VectorCircle`, `RasterCircle`, `VectorSquare`, etc. Add a third renderer and you have nine classes. Bridge splits this into two independent hierarchies: shapes hold a renderer, renderers know how to draw primitives, and the combinations multiply through composition rather than inheritance.

**The contrast with Adapter:** Adapter fixes an incompatibility at the boundary of code you cannot change. Bridge is a structural choice you make at design time to avoid a class explosion you can see coming.

**Python's take:** plain composition — pass the implementation into the abstraction's constructor.
**Kotlin's take:** the same, often with interface delegation (the `by` keyword).


### Python


In [ ]:
from abc import ABC, abstractmethod


# Implementation hierarchy — renderers know how to draw primitives.
class Renderer(ABC):
    @abstractmethod
    def render_circle(self, x: float, y: float, r: float) -> str: ...

    @abstractmethod
    def render_square(self, x: float, y: float, side: float) -> str: ...


class VectorRenderer(Renderer):
    def render_circle(self, x, y, r) -> str:
        return f"<svg circle cx={x} cy={y} r={r} />"

    def render_square(self, x, y, side) -> str:
        return f"<svg rect x={x} y={y} side={side} />"


class RasterRenderer(Renderer):
    def render_circle(self, x, y, r) -> str:
        return f"pixels: circle({x},{y},{r})"

    def render_square(self, x, y, side) -> str:
        return f"pixels: square({x},{y},{side})"


# Abstraction hierarchy — shapes hold a renderer.
class Shape(ABC):
    def __init__(self, renderer: Renderer):
        self._renderer = renderer

    @abstractmethod
    def draw(self) -> str: ...


class Circle(Shape):
    def __init__(self, renderer: Renderer, x: float, y: float, r: float):
        super().__init__(renderer)
        self.x, self.y, self.r = x, y, r

    def draw(self) -> str:
        return self._renderer.render_circle(self.x, self.y, self.r)


class Square(Shape):
    def __init__(self, renderer: Renderer, x: float, y: float, side: float):
        super().__init__(renderer)
        self.x, self.y, self.side = x, y, side

    def draw(self) -> str:
        return self._renderer.render_square(self.x, self.y, self.side)


# Mix and match — two shapes × two renderers = four combinations, no extra classes.
print(Circle(VectorRenderer(), 0, 0, 5).draw())
print(Circle(RasterRenderer(), 0, 0, 5).draw())
print(Square(VectorRenderer(), 0, 0, 10).draw())
print(Square(RasterRenderer(), 0, 0, 10).draw())


### Kotlin

```kotlin
interface Renderer {
    fun renderCircle(x: Double, y: Double, r: Double): String
    fun renderSquare(x: Double, y: Double, side: Double): String
}

class VectorRenderer : Renderer {
    override fun renderCircle(x: Double, y: Double, r: Double) = "<svg circle cx=$x cy=$y r=$r />"
    override fun renderSquare(x: Double, y: Double, side: Double) = "<svg rect x=$x y=$y side=$side />"
}

class RasterRenderer : Renderer {
    override fun renderCircle(x: Double, y: Double, r: Double) = "pixels: circle($x,$y,$r)"
    override fun renderSquare(x: Double, y: Double, side: Double) = "pixels: square($x,$y,$side)"
}

abstract class Shape(protected val renderer: Renderer) {
    abstract fun draw(): String
}

class Circle(renderer: Renderer, val x: Double, val y: Double, val r: Double) : Shape(renderer) {
    override fun draw() = renderer.renderCircle(x, y, r)
}

class Square(renderer: Renderer, val x: Double, val y: Double, val side: Double) : Shape(renderer) {
    override fun draw() = renderer.renderSquare(x, y, side)
}

println(Circle(VectorRenderer(), 0.0, 0.0, 5.0).draw())
println(Circle(RasterRenderer(), 0.0, 0.0, 5.0).draw())
```


**When not to use Bridge.** When you don't actually have two dimensions of variation. If shapes only ever render to one target, the renderer parameter is dead weight. Reach for Bridge when you can *see* the class explosion coming — when "how many subclasses would I need if I didn't split this?" answers with multiplication.


## Composite

**Intent:** compose objects into tree structures, and let clients treat individual objects and compositions of objects uniformly.

**Where it shows up:** file systems (file vs. directory), UI widget trees (button vs. panel), expression trees (literal vs. operator), organization charts (employee vs. manager), arithmetic ASTs.

**The shape:** a `Component` interface that both leaves and composites implement. Leaves do the actual work. Composites hold a list of children (each of which is itself a `Component`) and delegate operations down to them. The recursive structure means client code can call `total_size()` on a file or a directory holding ten thousand files — same call, same return type.

**Python's take:** a base class or protocol with the shared interface, plus a composite class that holds a list of children and forwards calls.

**Kotlin's take:** a `sealed interface` with leaf and composite subtypes, or a base class with composite holding a list.


### Python


In [ ]:
from abc import ABC, abstractmethod


class FsNode(ABC):
    @abstractmethod
    def size(self) -> int: ...

    @abstractmethod
    def render(self, indent: int = 0) -> str: ...


class File(FsNode):
    def __init__(self, name: str, size_bytes: int):
        self.name = name
        self._size = size_bytes

    def size(self) -> int:
        return self._size

    def render(self, indent: int = 0) -> str:
        return f"{'  ' * indent}- {self.name} ({self._size}B)"


class Directory(FsNode):
    def __init__(self, name: str, children: list[FsNode] | None = None):
        self.name = name
        self.children: list[FsNode] = children or []

    def add(self, child: FsNode) -> None:
        self.children.append(child)

    def size(self) -> int:
        return sum(c.size() for c in self.children)  # recurses through composites

    def render(self, indent: int = 0) -> str:
        head = f"{'  ' * indent}+ {self.name}/"
        body = "\n".join(c.render(indent + 1) for c in self.children)
        return head + ("\n" + body if body else "")


root = Directory("project", [
    File("README.md", 1200),
    Directory("src", [
        File("main.py", 3400),
        File("utils.py", 800),
    ]),
    Directory("tests", [
        File("test_main.py", 2100),
    ]),
])

print(root.size())     # 7500 — same call works on a file or a directory
print(root.render())


### Kotlin

```kotlin
sealed interface FsNode {
    fun size(): Int
    fun render(indent: Int = 0): String
}

class File(val name: String, val sizeBytes: Int) : FsNode {
    override fun size() = sizeBytes
    override fun render(indent: Int) = "  ".repeat(indent) + "- $name (${sizeBytes}B)"
}

class Directory(val name: String, val children: MutableList<FsNode> = mutableListOf()) : FsNode {
    fun add(child: FsNode) = children.add(child)
    override fun size() = children.sumOf { it.size() }
    override fun render(indent: Int): String {
        val head = "  ".repeat(indent) + "+ $name/"
        val body = children.joinToString("
") { it.render(indent + 1) }
        return if (body.isEmpty()) head else "$head
$body"
    }
}

val root = Directory("project", mutableListOf(
    File("README.md", 1200),
    Directory("src", mutableListOf(
        File("main.py", 3400),
        File("utils.py", 800),
    )),
))

println(root.size())
println(root.render())
```


**When not to use Composite.** When the structure is not actually recursive. If "groups of things" always contain only single items (no nested groups), you don't need a composite — a list is enough. Composite earns its keep when the tree can be arbitrarily deep and client code wants to treat one node and a thousand-node subtree identically.


## Decorator

**Intent:** attach additional responsibilities to an object dynamically, providing a flexible alternative to subclassing for extending behaviour.

**Where it shows up:** I/O streams (buffered, compressed, encrypted layers on a raw stream), HTTP middleware chains, logging/timing/caching wrappers on a function or service, feature toggles.

**The shape:** a wrapper that has the same interface as the wrapped object, plus extra behaviour. The wrapper forwards calls to the inner object and adds its own logic before or after. Wrappers can stack — a `Cached` wrapping a `Logged` wrapping a real service.

**Python's take:** the `@decorator` language feature *is* this pattern, applied to functions. For object decoration, write a class that forwards to the wrapped instance and adds behaviour. `functools.wraps` and `functools.cache` are decorator-pattern instances built into the standard library.

**Kotlin's take:** interface delegation with `by` makes decorators almost free — you delegate every method to a wrapped instance and override only the ones you want to decorate.


### Python


In [ ]:
import time
from typing import Protocol


class DataSource(Protocol):
    def fetch(self, key: str) -> str: ...


class SlowDataSource:
    def fetch(self, key: str) -> str:
        time.sleep(0.01)  # imagine a real network call
        return f"value-for-{key}"


# Decorator 1 — adds caching.
class CachedDataSource:
    def __init__(self, inner: DataSource):
        self._inner = inner
        self._cache: dict[str, str] = {}

    def fetch(self, key: str) -> str:
        if key not in self._cache:
            self._cache[key] = self._inner.fetch(key)
        return self._cache[key]


# Decorator 2 — adds logging.
class LoggedDataSource:
    def __init__(self, inner: DataSource):
        self._inner = inner

    def fetch(self, key: str) -> str:
        print(f"  fetch({key!r})")
        return self._inner.fetch(key)


# Stack them — logging wraps caching wraps the real source.
source = LoggedDataSource(CachedDataSource(SlowDataSource()))

print(source.fetch("a"))   # logs, cache miss, slow call
print(source.fetch("a"))   # logs, cache hit, fast
print(source.fetch("b"))   # logs, cache miss


### Kotlin

```kotlin
interface DataSource {
    fun fetch(key: String): String
}

class SlowDataSource : DataSource {
    override fun fetch(key: String): String {
        Thread.sleep(10)
        return "value-for-$key"
    }
}

// Decorator via interface delegation — `by inner` forwards every method
// to `inner` automatically. We only override fetch to add caching.
class CachedDataSource(private val inner: DataSource) : DataSource by inner {
    private val cache = mutableMapOf<String, String>()
    override fun fetch(key: String): String =
        cache.getOrPut(key) { inner.fetch(key) }
}

class LoggedDataSource(private val inner: DataSource) : DataSource by inner {
    override fun fetch(key: String): String {
        println("  fetch($key)")
        return inner.fetch(key)
    }
}

val source: DataSource = LoggedDataSource(CachedDataSource(SlowDataSource()))
println(source.fetch("a"))
println(source.fetch("a"))   // cache hit
println(source.fetch("b"))
```


**When not to use Decorator.** When the wrapped behaviour is single-purpose and you don't need to stack capabilities. If you only ever add caching, a function with a `@functools.cache` decorator is shorter than a class hierarchy. Decorator earns its keep when several capabilities (logging, caching, retries, rate limiting) need to be combinable in any order.


## Facade

**Intent:** provide a unified, simplified interface to a set of interfaces in a subsystem.

**Where it shows up:** SDK design (a thin convenience class wrapping a complex client library), service layers (one method that orchestrates four repository calls), "convenience" classes (`shutil.make_archive` hiding `tarfile` + `gzip` + file walking).

**The shape:** the facade is a single class whose methods kick off complex sequences of calls into the subsystem. Clients import the facade and never touch the underlying machinery directly.

**The contrast with Adapter:** an adapter exists because the subsystem's interface is *wrong*. A facade exists because the subsystem's interface is *too complex*. Adapter changes shape; facade reduces surface area.

**Python's take:** a class — or even just a module-level function — that hides the orchestration.

**Kotlin's take:** the same. Often a single class with a few methods that call into several injected dependencies.


### Python


In [ ]:
# A "complex subsystem" — three small classes that each do one thing.
class VideoFile:
    def __init__(self, path: str):
        self.path = path
        self.codec = "mp4"


class CodecFactory:
    def extract(self, video: VideoFile) -> str:
        return f"{video.codec}-stream"


class AudioMixer:
    def fix(self, stream: str) -> str:
        return f"mixed[{stream}]"


class BitrateReader:
    def read(self, stream: str) -> str:
        return f"compressed[{stream}]"


# The facade — one method hides the whole pipeline.
class VideoConverter:
    def convert(self, source_path: str, target_format: str) -> str:
        video = VideoFile(source_path)
        stream = CodecFactory().extract(video)
        stream = AudioMixer().fix(stream)
        stream = BitrateReader().read(stream)
        return f"{stream}.{target_format}"


# Client code stays trivial.
result = VideoConverter().convert("input.mp4", "ogg")
print(result)  # compressed[mixed[mp4-stream]].ogg


### Kotlin

```kotlin
class VideoFile(val path: String) { val codec = "mp4" }
class CodecFactory { fun extract(v: VideoFile) = "${v.codec}-stream" }
class AudioMixer { fun fix(s: String) = "mixed[$s]" }
class BitrateReader { fun read(s: String) = "compressed[$s]" }

class VideoConverter {
    fun convert(sourcePath: String, targetFormat: String): String {
        val video = VideoFile(sourcePath)
        var stream = CodecFactory().extract(video)
        stream = AudioMixer().fix(stream)
        stream = BitrateReader().read(stream)
        return "$stream.$targetFormat"
    }
}

println(VideoConverter().convert("input.mp4", "ogg"))
```


**When not to use Facade.** When the subsystem is already simple — a facade over a one-method API is just a longer name for the same thing. Facade earns its keep when callers consistently need *the same multi-step orchestration*, and that orchestration is currently duplicated at every call site.


## Flyweight

**Intent:** use sharing to support large numbers of fine-grained objects efficiently.

**Where it shows up:** rendering ten thousand particles in a game (one shared sprite, ten thousand positions), character glyphs in a text editor (one glyph object per character, regardless of how many times it appears on screen), tile maps, syntax highlighting (one style object per token type).

**The key distinction:** **intrinsic state** is shared across all uses of the flyweight (the glyph's shape, the particle's sprite). **Extrinsic state** is unique per use and lives outside the flyweight (the glyph's screen position, the particle's velocity). The pattern works only when the shared state dwarfs the per-instance state — otherwise sharing buys nothing.

**Python's take:** a cache (often a dictionary) of flyweight instances, keyed by the intrinsic state. `__slots__` to keep memory tight.

**Kotlin's take:** a companion object cache, or — if the set is small and closed — an `enum`.


### Python


In [ ]:
from dataclasses import dataclass


# Intrinsic state — shared across all uses of the same glyph shape.
@dataclass(frozen=True, slots=True)
class Glyph:
    char: str
    font: str
    size: int


# Flyweight factory — guarantees one Glyph instance per (char, font, size) triple.
class GlyphFactory:
    def __init__(self):
        self._cache: dict[tuple[str, str, int], Glyph] = {}

    def get(self, char: str, font: str, size: int) -> Glyph:
        key = (char, font, size)
        if key not in self._cache:
            self._cache[key] = Glyph(char, font, size)
        return self._cache[key]

    def cached_count(self) -> int:
        return len(self._cache)


# Extrinsic state — position lives outside the flyweight, in the document.
@dataclass(slots=True)
class Character:
    glyph: Glyph     # shared reference
    x: int
    y: int


factory = GlyphFactory()

# Render the word "look" twice — same glyphs reused.
document = []
for line, word in enumerate(["look", "look"]):
    for col, ch in enumerate(word):
        document.append(Character(factory.get(ch, "Inter", 14), col * 8, line * 18))

print(f"characters in document: {len(document)}")           # 8
print(f"distinct glyphs cached: {factory.cached_count()}")  # 3  (l, o, k — 'o' reused)
print(document[1].glyph is document[5].glyph)               # True — same Glyph object


### Kotlin

```kotlin
data class Glyph(val char: Char, val font: String, val size: Int)

object GlyphFactory {
    private val cache = mutableMapOf<Triple<Char, String, Int>, Glyph>()
    fun get(char: Char, font: String, size: Int): Glyph =
        cache.getOrPut(Triple(char, font, size)) { Glyph(char, font, size) }
    fun cachedCount() = cache.size
}

data class Character(val glyph: Glyph, val x: Int, val y: Int)

val document = mutableListOf<Character>()
listOf("look", "look").forEachIndexed { line, word ->
    word.forEachIndexed { col, ch ->
        document.add(Character(GlyphFactory.get(ch, "Inter", 14), col * 8, line * 18))
    }
}

println("characters in document: ${document.size}")
println("distinct glyphs cached: ${GlyphFactory.cachedCount()}")
println(document[1].glyph === document[5].glyph)  // true — same reference
```


**When not to use Flyweight.** When you have *thousands* but not *millions* — the cache overhead may exceed the savings. Profile before reaching for this. Modern JVMs and CPython are aggressive about deduplicating short strings and small integers already, so a naïve flyweight on top of those is often pointless. Flyweight earns its keep when the intrinsic state is genuinely large (a sprite, a complex glyph, a syntax style) and the duplication count is genuinely huge.


## Proxy

**Intent:** provide a surrogate or placeholder for another object to control access to it.

**Where it shows up:** lazy loading (don't construct an expensive resource until first use), access control (authorize before forwarding), caching (return memoized result instead of calling through), remote method invocation (proxy on the client, real object on the server), virtual proxies for ORM lazy collections.

**Four flavours of proxy from the GoF book:**

- **Virtual proxy** — defers expensive object creation until first use.
- **Protection proxy** — adds an access-control check before forwarding.
- **Remote proxy** — represents an object that lives in another process/machine.
- **Smart reference** — adds reference counting, locking, or logging on access.

**The contrast with Decorator:** decorator *adds capabilities* to an object that the caller wants. Proxy *controls access* to an object the caller would otherwise use directly. Same structural shape, different intent.

**Python's take:** `__getattr__` for transparent proxies; explicit wrapper class for the four flavours.
**Kotlin's take:** interface delegation with `by`, with overrides for the access-control logic.


### Python


In [ ]:
from typing import Protocol


class Image(Protocol):
    def render(self) -> str: ...


# The "real" object — expensive to construct.
class RealImage:
    def __init__(self, path: str):
        print(f"  [loading {path} from disk]")  # imagine a slow I/O call
        self._path = path
        self._pixels = f"<pixels of {path}>"

    def render(self) -> str:
        return self._pixels


# Virtual proxy — defers construction until render() is actually called.
class LazyImage:
    def __init__(self, path: str):
        self._path = path
        self._real: RealImage | None = None

    def render(self) -> str:
        if self._real is None:
            self._real = RealImage(self._path)
        return self._real.render()


# Protection proxy — checks permissions before delegating.
class ProtectedImage:
    def __init__(self, inner: Image, user_role: str):
        self._inner = inner
        self._role = user_role

    def render(self) -> str:
        if self._role != "admin":
            raise PermissionError("admin only")
        return self._inner.render()


# Construct ten lazy proxies — no disk I/O happens yet.
gallery = [LazyImage(f"photo_{i}.jpg") for i in range(10)]
print("gallery built — note: no [loading] messages above")

# Only the rendered one actually loads.
print(gallery[3].render())

# Add a protection proxy on top.
protected = ProtectedImage(LazyImage("secret.jpg"), user_role="guest")
try:
    protected.render()
except PermissionError as e:
    print(f"blocked: {e}")


### Kotlin

```kotlin
interface Image { fun render(): String }

class RealImage(private val path: String) : Image {
    init { println("  [loading $path from disk]") }
    private val pixels = "<pixels of $path>"
    override fun render() = pixels
}

// Virtual proxy
class LazyImage(private val path: String) : Image {
    private val real by lazy { RealImage(path) }
    override fun render() = real.render()
}

// Protection proxy
class ProtectedImage(private val inner: Image, private val role: String) : Image {
    override fun render(): String {
        require(role == "admin") { "admin only" }
        return inner.render()
    }
}

val gallery = (0..9).map { LazyImage("photo_$it.jpg") }
println("gallery built — no loads yet")
println(gallery[3].render())

val protected = ProtectedImage(LazyImage("secret.jpg"), role = "guest")
try { protected.render() } catch (e: IllegalArgumentException) { println("blocked: ${e.message}") }
```


**When not to use Proxy.** When you don't need *control* over access — only added capability. That's Decorator, not Proxy. Both wrap an object with the same interface; the distinction is purely about intent. Proxy is also overkill when the wrapped object is cheap to construct and freely accessible — lazy loading a small object, or access-controlling a public one, just adds an indirection layer for no benefit.


## Picking the right structural pattern

| Question | Pattern |
|---|---|
| "I need to make a class with the wrong interface usable through the right one." | **Adapter** |
| "I have two dimensions of variation and don't want a class explosion." | **Bridge** |
| "I need to treat one thing and a tree of things uniformly." | **Composite** |
| "I want to layer behaviour on an object, stackably." | **Decorator** |
| "I want one simple entry point to a complicated subsystem." | **Facade** |
| "I have huge numbers of similar objects and need to share their shared state." | **Flyweight** |
| "I need to control access to another object — lazy, protected, remote, smart." | **Proxy** |

Three of these blur into each other and are worth distinguishing one more time.

- **Adapter** *converts* an interface. The wrapped object's shape was wrong; the adapter makes it right.
- **Decorator** *adds capabilities*. The wrapped object's shape is fine; the decorator extends it.
- **Proxy** *controls access*. The wrapped object's shape is fine; the proxy stands in front of it.

Same structural pattern (wrap an object, forward calls), three different intents. As always, name the pattern by its intent, not its shape.


Notebook four turns to **Behavioral patterns — Core**. The five behavioral patterns you will reach for most often in real code: Strategy, Observer, Command, Template Method, and Iterator. These five carry most of the day-to-day weight, which is why they get their own notebook before we get to the more situational five in notebook 05.
